# N5 — Narrativa cultural (Visualización)
## AI-MELT: Dashboard final del campo metafórico

**Prerequisito:** Haber ejecutado todos los notebooks de procesamiento N0 a N5.

---

### Archivos de entrada esperados

| Directorio | Archivo | Descripción |
|---|---|---|
| `./outputs/N5/` | `n5_narrativa_cultural.csv` | Narrativa dominante con score y métricas |
| `./outputs/N5/` | `n5_narrativas_alternativas.csv` | Todas las narrativas: dominante, challenger, emergente |
| `./outputs/N5/` | `n5_resumen_campo_metaforico.json` | Resumen completo para dashboard |
| `./outputs/N4/` | `n4_regimenes.csv`, `n4_escenario_regimen.csv`, `n4_grafo_cache.json` | Regímenes y grafo |
| `./outputs/N3/` | `n3_escenarios.csv` | Escenarios |
| `./outputs/N2/` | `n2_metaforas_convencionales.csv` | Metáforas convencionales |
| `./outputs/N1/` | `n1_metaforas_primarias.csv` | Metáforas primarias |

---

### Visualizaciones generadas (Dashboard final)
1. Grafo del campo metafórico con régimen dominante resaltado
2. Sankey multinivel completo: N1 → N2 → N3 → N4 → N5
3. Tabla comparativa de narrativas (dominante vs. alternativas)
4. Radar chart: métricas del régimen dominante vs. otros
5. Resumen textual generado automáticamente

## 1. Configuración y carga

In [ ]:
# ============================================================
# 1. IMPORTS Y CARGA
# ============================================================
import json
import os
import warnings

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

N1_DIR, N2_DIR, N3_DIR, N4_DIR, N5_DIR = [f"./outputs/N{i}/" for i in range(6)]
VIZ_DIR = "./outputs/N5/viz/"
os.makedirs(VIZ_DIR, exist_ok=True)

# Cargar datos
df_narr_dom = pd.read_csv(os.path.join(N5_DIR, "n5_narrativa_cultural.csv"))
df_narr_all = pd.read_csv(os.path.join(N5_DIR, "n5_narrativas_alternativas.csv"))

with open(os.path.join(N5_DIR, "n5_resumen_campo_metaforico.json"), encoding='utf-8') as f:
    resumen = json.load(f)

df_reg = pd.read_csv(os.path.join(N4_DIR, "n4_regimenes.csv"))
df_esc_reg = pd.read_csv(os.path.join(N4_DIR, "n4_escenario_regimen.csv"))
df_esc = pd.read_csv(os.path.join(N3_DIR, "n3_escenarios.csv"))
df_conv = pd.read_csv(os.path.join(N2_DIR, "n2_metaforas_convencionales.csv"))
df_n1 = pd.read_csv(os.path.join(N1_DIR, "n1_metaforas_primarias.csv"))

with open(os.path.join(N4_DIR, "n4_grafo_cache.json"), encoding='utf-8') as f:
    grafo_data = json.load(f)

print("✓ Datos cargados:")
print(f"  Narrativas: {len(df_narr_all)}")
print(f"  Regímenes: {len(df_reg)}")
print(f"  Escenarios: {len(df_esc)}")
print(f"  Convencionales: {len(df_conv)}")
print(f"  Primarias: {len(df_n1):,}")

## 2. Grafo del campo metafórico con régimen dominante resaltado

In [ ]:
# ============================================================
# 2. GRAFO CON RÉGIMEN DOMINANTE RESALTADO
# ============================================================

G = nx.Graph()
for node in grafo_data["nodes"]:
    G.add_node(node["id"], **{k: v for k, v in node.items() if k != "id"})
for edge in grafo_data["edges"]:
    G.add_edge(edge["source"], edge["target"], weight=edge["weight"])

regime_map = dict(zip(df_esc_reg["ID_escenario"], df_esc_reg["ID_regimen"], strict=False))
dominant_rid = df_narr_dom.iloc[0]["ID_regimen_dominante"] if len(df_narr_dom) > 0 else ""

n_regimes = len(df_reg)
regime_colors = plt.cm.get_cmap("tab10", max(n_regimes, 1))
regime_id_to_idx = {r: i for i, r in enumerate(df_reg["ID_regimen"])}

fig, ax = plt.subplots(figsize=(18, 14))
pos_layout = nx.spring_layout(G, seed=42, k=2, weight="weight")

# Aristas
nx.draw_networkx_edges(G, pos_layout, alpha=0.1, width=0.5, ax=ax)

# Nodos
for node in G.nodes():
    reg_id = regime_map.get(node, "")
    reg_idx = regime_id_to_idx.get(reg_id, -1)
    is_dominant = reg_id == dominant_rid
    color = regime_colors(reg_idx) if reg_idx >= 0 else "lightgray"
    size = 300 + G.nodes[node].get("degree_centrality", 0) * 3000
    edge_color = "gold" if is_dominant else "white"
    edge_width = 3 if is_dominant else 0.5
    ax.scatter(*pos_layout[node], c=[color], s=size, alpha=0.9 if is_dominant else 0.6,
               edgecolors=edge_color, linewidth=edge_width, zorder=3)

labels = {n: G.nodes[n].get("label", n)[:18] for n in G.nodes()}
nx.draw_networkx_labels(G, pos_layout, labels, font_size=6, ax=ax)

# Leyenda
legend_patches = []
for i, (_, row) in enumerate(df_reg.iterrows()):
    is_dom = row["ID_regimen"] == dominant_rid
    label = f"{'👑 ' if is_dom else ''}{row['ID_regimen']}: {row['nombre_regimen'][:30]}"
    legend_patches.append(mpatches.Patch(facecolor=regime_colors(i), label=label,
                                          edgecolor="gold" if is_dom else "gray",
                                          linewidth=2 if is_dom else 0.5))
ax.legend(handles=legend_patches, loc='upper left', fontsize=7, title="Regímenes (👑 = dominante)")

dom_name = df_narr_dom.iloc[0]["nombre_narrativa"] if len(df_narr_dom) > 0 else "?"
ax.set_title(f"Campo metafórico — Narrativa dominante: {dom_name}", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_grafo_narrativa_dominante.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 3. Sankey multinivel completo: primarias → convencionales → escenarios → regímenes → narrativa

In [ ]:
# ============================================================
# 3. SANKEY MULTINIVEL N1→N2→N3→N4→N5 (Top elements)
# ============================================================

# Para legibilidad: solo top 15 convencionales, sus escenarios, regímenes y narrativa
top_mc = df_conv.sort_values("frecuencia_absoluta", ascending=False).head(15)
top_mc_ids = top_mc["ID_metaconvencional"].tolist()
top_mc_names = top_mc["metafora_conceptual"].tolist()

# Escenarios de esas convencionales
esc_of_top = df_esc[df_esc["ID_metaconvencional_base"].isin(top_mc_ids)]
esc_ids_top = esc_of_top["ID_escenario"].tolist()
esc_names_top = esc_of_top["nombre_escenario"].tolist()

# Regímenes de esos escenarios
reg_of_top = df_esc_reg[df_esc_reg["ID_escenario"].isin(esc_ids_top)]
reg_ids_top = reg_of_top["ID_regimen"].unique().tolist()
reg_names = {r["ID_regimen"]: r["nombre_regimen"] for _, r in df_reg.iterrows()}

# Narrativas de esos regímenes
narr_of_top = df_narr_all[df_narr_all["ID_regimen"].isin(reg_ids_top)]

# Build labels
all_labels = (
    [f"[MC] {m[:35]}" for m in top_mc_names] +
    [f"[ESC] {e[:30]}" for e in esc_names_top] +
    [f"[REG] {reg_names.get(r, r)[:30]}" for r in reg_ids_top] +
    [f"[NAR] {n[:30]}" for _, n in narr_of_top[["nombre_narrativa"]].drop_duplicates().iterrows()
     for n in [n["nombre_narrativa"]]]
)
# Deduplicate
all_labels = list(dict.fromkeys(all_labels))
node_idx = {lbl: i for i, lbl in enumerate(all_labels)}

sources, targets, values, colors = [], [], [], []

# MC → ESC
for _, esc_row in esc_of_top.iterrows():
    mc_lbl = f"[MC] {df_conv[df_conv['ID_metaconvencional'] == esc_row['ID_metaconvencional_base']]['metafora_conceptual'].iloc[0][:35]}" if esc_row["ID_metaconvencional_base"] in top_mc_ids else None
    esc_lbl = f"[ESC] {esc_row['nombre_escenario'][:30]}"
    if mc_lbl and mc_lbl in node_idx and esc_lbl in node_idx:
        sources.append(node_idx[mc_lbl])
        targets.append(node_idx[esc_lbl])
        values.append(1)
        colors.append("rgba(150,150,200,0.3)")

# ESC → REG
for _, er in reg_of_top.iterrows():
    esc_name = df_esc[df_esc["ID_escenario"] == er["ID_escenario"]]["nombre_escenario"]
    if len(esc_name) > 0:
        esc_lbl = f"[ESC] {esc_name.iloc[0][:30]}"
        reg_lbl = f"[REG] {reg_names.get(er['ID_regimen'], '')[:30]}"
        if esc_lbl in node_idx and reg_lbl in node_idx:
            sources.append(node_idx[esc_lbl])
            targets.append(node_idx[reg_lbl])
            values.append(1)
            is_dom = er["ID_regimen"] == dominant_rid
            colors.append("rgba(29,158,117,0.5)" if is_dom else "rgba(150,150,150,0.3)")

# REG → NAR
for _, nr in narr_of_top.iterrows():
    reg_lbl = f"[REG] {reg_names.get(nr['ID_regimen'], '')[:30]}"
    nar_lbl = f"[NAR] {nr['nombre_narrativa'][:30]}"
    if reg_lbl in node_idx and nar_lbl in node_idx:
        sources.append(node_idx[reg_lbl])
        targets.append(node_idx[nar_lbl])
        values.append(3)
        is_dom = nr.get("tipo", "") == "dominante"
        colors.append("rgba(29,158,117,0.7)" if is_dom else "rgba(200,150,50,0.5)")

if sources:
    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=8, thickness=12, line=dict(color="black", width=0.5), label=all_labels),
        link=dict(source=sources, target=targets, value=values, color=colors)
    )])
    fig.update_layout(
        title="Sankey multinivel: metáforas convencionales → escenarios → regímenes → narrativas",
        font_size=9, height=800
    )
    pio.write_html(fig, os.path.join(VIZ_DIR, "viz_sankey_multinivel_completo.html"), include_plotlyjs="cdn")
    print("✓ viz_sankey_multinivel_completo.html")
    fig.show()
else:
    print("⚠ Sin datos suficientes para Sankey multinivel.")

## 4. Tabla comparativa de narrativas

In [ ]:
# ============================================================
# 4. TABLA COMPARATIVA (Plotly)
# ============================================================

if len(df_narr_all) > 0:
    display_cols = ["tipo", "nombre_narrativa", "score", "orientacion_comportamiento"]
    display_cols = [c for c in display_cols if c in df_narr_all.columns]
    df_display = df_narr_all[display_cols].copy()

    # Truncar textos largos
    for col in df_display.columns:
        df_display[col] = df_display[col].astype(str).str[:200]

    df_display.columns = [c.replace("_", " ").title() for c in display_cols]

    tipo_colors = {"dominante": "#1D9E7544", "challenger": "#E8A83844", "emergente": "#3B8BD444", "periferico": "#99999944"}
    cell_fills = [[tipo_colors.get(str(t).lower(), "#ffffff") for t in df_narr_all["tipo"]]]

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(df_display.columns), fill_color='#1F4E79',
                     font=dict(color='white', size=11), align='left'),
        cells=dict(values=[df_display[c] for c in df_display.columns],
                    font=dict(size=10), align='left', height=30,
                    fill_color=[cell_fills[0]] * len(df_display.columns))
    )])
    fig.update_layout(title="Narrativas culturales — Comparación", height=max(300, 50*len(df_narr_all)))
    pio.write_html(fig, os.path.join(VIZ_DIR, "viz_tabla_narrativas.html"), include_plotlyjs="cdn")
    print("✓ viz_tabla_narrativas.html")
    fig.show()

## 5. Radar chart: métricas del régimen dominante vs. otros

In [ ]:
# ============================================================
# 5. RADAR CHART DE REGÍMENES
# ============================================================

ranking = resumen.get("ranking_regimenes", [])

if ranking:
    categories = ["Frecuencia", "Coherencia", "Dispersión", "Score"]
    tipo_colors = {"dominante": "#1D9E75", "challenger": "#E8A838", "emergente": "#3B8BD4", "periferico": "#999999"}

    fig = go.Figure()
    max_freq = max(r.get("frecuencia_agregada", 1) for r in ranking) or 1

    for r in ranking[:5]:  # Top 5
        values = [
            r.get("frecuencia_agregada", 0) / max_freq,
            r.get("coherencia_interna", 0),
            r.get("dispersion_textual", 0),
            r.get("score", 0),
        ]
        values.append(values[0])

        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories + [categories[0]],
            fill='toself',
            name=f"{r.get('tipo', '').upper()}: {r.get('nombre_regimen', '')[:30]}",
            opacity=0.6,
            line=dict(color=tipo_colors.get(r.get("tipo", ""), "#888"), width=3 if r.get("tipo") == "dominante" else 1)
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Radar: métricas de regímenes (dominante vs. alternativas)",
        height=600
    )
    pio.write_html(fig, os.path.join(VIZ_DIR, "viz_radar_regimenes.html"), include_plotlyjs="cdn")
    print("✓ viz_radar_regimenes.html")
    fig.show()

## 6. Resumen textual del campo metafórico

In [ ]:
# ============================================================
# 6. RESUMEN TEXTUAL AUTOMÁTICO
# ============================================================

stats = resumen.get("estadisticas_generales", {})
dom = resumen.get("narrativa_dominante", {})

print("╔" + "═"*68 + "╗")
print("║  RESUMEN DEL CAMPO METAFÓRICO — AI-MELT" + " "*27 + "║")
print("║  Informe Final de la Comisión de la Verdad de Colombia" + " "*12 + "║")
print("╚" + "═"*68 + "╝")
print()
print(f"  Metáforas primarias identificadas:  {stats.get('metaforas_primarias', '?'):,}")
print(f"  Metáforas convencionalizadas:       {stats.get('metaforas_convencionales', '?')}")
print(f"  Escenarios metafóricos construidos: {stats.get('escenarios', '?')}")
print(f"  Regímenes de metáforas:             {stats.get('regimenes', '?')}")
print(f"  Narrativas culturales formuladas:   {stats.get('narrativas_generadas', '?')}")
print()
print("─"*70)
print()

if dom:
    print(f"  👑 NARRATIVA DOMINANTE: {dom.get('nombre_narrativa', '?')}")
    print(f"     Score: {dom.get('score', 0):.3f}")
    print()
    desc = dom.get("descripcion_narrativa", "")
    # Imprimir descripción con word wrap
    import textwrap
    for line in textwrap.wrap(desc, width=65):
        print(f"     {line}")
    print()
    vals_leg = dom.get("valores_legitimados", [])
    vals_des = dom.get("valores_deslegitimados", [])
    emos_fac = dom.get("emociones_facilitadas", [])
    emos_inh = dom.get("emociones_inhibidas", [])
    if vals_leg:
        print(f"     Valores legitimados:     {', '.join(vals_leg)}")
    if vals_des:
        print(f"     Valores deslegitimados:  {', '.join(vals_des)}")
    if emos_fac:
        print(f"     Emociones facilitadas:   {', '.join(emos_fac)}")
    if emos_inh:
        print(f"     Emociones inhibidas:     {', '.join(emos_inh)}")
    print()
    orient = dom.get("orientacion_comportamiento", "")
    if orient:
        print("     Orientación del comportamiento colectivo:")
        for line in textwrap.wrap(orient, width=65):
            print(f"     {line}")

print()
print("─"*70)
print()

# Alternativas
alts = resumen.get("narrativas_alternativas", [])
for alt in alts:
    print(f"  {'⚔️' if alt.get('tipo')=='challenger' else '🌱'} {alt.get('tipo', '').upper()}: {alt.get('nombre_narrativa', '?')}")
    print(f"     Score: {alt.get('score', 0):.3f}")
    desc_alt = alt.get("descripcion_narrativa", "")
    if desc_alt:
        for line in textwrap.wrap(desc_alt[:300], width=65):
            print(f"     {line}")
    print()

## 7. Resumen

In [ ]:
# ============================================================
# 7. RESUMEN
# ============================================================
print("=" * 60)
print("N5 — DASHBOARD FINAL COMPLETADO")
print("=" * 60)
print(f"\nArchivos en {VIZ_DIR}:")
for f_name in sorted(os.listdir(VIZ_DIR)):
    size = os.path.getsize(os.path.join(VIZ_DIR, f_name)) / 1024
    icon = "📊" if f_name.endswith(".html") else "🖼"
    print(f"  {icon} {f_name} ({size:.0f} KB)")
print()
print("═"*60)
print("  PIPELINE AI-MELT COMPLETADO")
print("  N0 → N1 → N2 → N3 → N4 → N5")
print("  Del texto a la narrativa cultural")
print("═"*60)